# Aula 1, Generative Models (AI Engineering)
## Representações de texto, da contagem de palavras ao modelo pré-treinado

Cinco demonstrações, na ordem em que entram na aula. Cada uma imprime um número medido pela própria célula. Rode as células em ordem.

A célula seguinte instala as dependências e leva cerca de um minuto no Colab.

In [ ]:
!pip install stop-words huggingface_hub safetensors gensim anthropic --quiet

---
## 1. Bag of words, filtro de spam

Cada mensagem vira um vetor de contagem de palavras, com uma posição por palavra do vocabulário do corpus. O corpus tem 50 mensagens em português, 25 marcadas como spam e 25 como normais, adaptadas do SMS Spam Collection (UCI) com o vocabulário de golpe e de venda do original. O classificador é um Naive Bayes multinomial, avaliado por validação cruzada em cinco dobras.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

spam = [
    "Você é um dos sorteados! Foi selecionado para receber R$5.000 em dinheiro ou um pacote de viagem com passagens incluídas. Fale com um atendente agora para resgatar.",
    "PRIVADO! Seu extrato de 2023 da linha mostra 800 pontos não resgatados. Ligue agora e informe o código de identificação 40533. Oferta expira em 31/10.",
    "5 conteúdos digitais grátis! Ligue agora e receba um conteúdo novo toda semana por apenas R$15 por mês. Para cancelar, envie SAIR.",
    "Já tem o celular há 10 meses? Atualize para o novo modelo com câmera de graça. Economize com ligações e mensagens de fim de semana grátis. Responda SIM para receber uma ligação.",
    "Você ganhou R$5.000 em dinheiro ou um prêmio de R$10.000! Para resgatar, ligue agora para o número informado.",
    "Ganhe o novo smartphone ou tocador de música, prêmio de R$2.000 garantido! Responda GANHEI agora mesmo.",
    "Ligações internacionais a partir de R$0,10 por minuto pela nossa operadora. Consulte condições no site. Para sair da lista, responda SAIR.",
    "Ligue para nosso atendimento entre 10h e 21h, pois você GANHOU um prêmio garantido de R$5.000 em dinheiro!",
    "Toc toc! Responda quem é para participar do sorteio semanal de um vale-compras de R$1.250 na loja de sua escolha.",
    "Baixe quantos toques quiser, sem limite, milhares de opções. Você pode até enviar para os amigos. Envie um código e pague apenas R$15.",
    "Quer um celular novo com câmera de vídeo? Minutos ilimitados, chamadas de vídeo incluídas e downloads liberados toda semana, de graça e com entrega amanhã. Ligue agora ou responda AGORA.",
    "Já tem o celular há 11 meses ou mais? Você tem direito a atualizar para os modelos mais novos com câmera, de graça! Ligue para a Central de Atualização, é grátis.",
    "Mensagem grátis! Baixe agora o novo toque que todo mundo está usando. Seis sons diferentes por apenas R$15 por semana. Só para maiores de 16 anos.",
    "Teve o crédito recusado? Com ou sem garantia, não consegue crédito? Ligue grátis agora ou responda AJUDA que a gente resolve.",
    "GANHADOR!! Como cliente valioso da nossa rede, você foi selecionado para receber um prêmio de R$4.500! Para resgatar, ligue e informe o código KL341. Válido só por 12 horas.",
    "Dobre seus minutos e mensagens ou pague metade no plano. Aparelhos com Bluetooth de graça. Ligue para atualizar seu plano ou responda SAIR para cancelar.",
    "Parabéns! Dois celulares com câmera de vídeo são seus. Ligue agora, converse por vídeo com os amigos, jogue online e baixe música, sem taxa de linha.",
    "Temos novos contatos cadastrados recentemente na sua região. Responda AGORA para começar a conversar.",
    "Primeira semana de inscrição grátis! Concorra a um prêmio em dinheiro toda semana. Envie um código para participar. Consulte o regulamento no site.",
    "PESSOAL! Estamos tentando falar com você. O sorteio do fim de semana mostrou que você ganhou um prêmio garantido de R$5.000. Ligue e informe o código K52. Válido só por 12 horas.",
    "Seu saldo em dinheiro é de R$2.500 atualmente. Para resgatar agora, envie a palavra RESGATE para o número informado.",
    "Enquete rápida: o time da casa venceu o jogo de ontem? Responda CERTO ou ERRADO para participar.",
    "Inscrição grátis no sorteio semanal para ganhar ingressos para a final do campeonato ou R$1.500 em dinheiro. Envie um código para participar. Só para maiores de 18 anos.",
    "Você tem o que é preciso para participar do rali? Participe com a nossa marca parceira! Envie um código para se inscrever e ver os detalhes no site.",
    "Seu extrato mostra 800 pontos não resgatados na sua conta. Ligue agora e informe o código de identificação 42478. Oferta expira em breve.",
]

ham = [
    "Kkkk ok, não vou mais precisar que você faça meu cabelo.",
    "Tunde, como você está. Só passando para desejar um ótimo dia. Abiola.",
    "Que ótimo. Meu vizinho vem pegar pasta de dente toda manhã.",
    "Kkk, ai, mas eu queria ter ficado fora um pouco mais.",
    "Você ainda está por aí? Posso passar aí mais tarde.",
    "Vi um musical muito bom ontem à noite.",
    "Gente, R$700 ou R$900 por 5 noites, ótima localização com café da manhã incluso.",
    "Está ensolarado aqui. O clima está agradável.",
    "Te encontro na frente da loja, você vai ver como minha cabeça funciona.",
    "Não consigo atender o telefone agora. Manda uma mensagem.",
    "Ok, tudo certo. É logo depois da rua tal, atrás do campo. Me liga quando estiver por perto, te vejo na terça.",
    "Hoje vou para a faculdade, então não vou conseguir ir à aula.",
    "Ah, você também não sabe. Outro dia fiz uma coisa simples e esperta com pera, perfeita para o Natal.",
    "Isso tudo é meio estranho e maluco para mim.",
    "Haha eu ouvi isso, me manda mensagem quando você estiver por aí.",
    "Você não vai ficar chateada quando eu usar o dinheiro dela para te levar para jantar um bife.",
    "Não tenho certeza, eu só estava vendo o que estava acontecendo por aqui.",
    "Beleza, posso passar aí à noite se minha aula terminar mais cedo.",
    "Ok, só estava brincando com você.",
    "Não, acho que ele não estuda lá, mas mora aqui perto.",
    "Se você perguntar para ela, ou se ela falar alguma coisa, me avisa.",
    "Guarda um lugar para mim no seu coração, porque a mente esquece mas o coração sempre lembra. Feliz Dia dos Namorados.",
    "Manda uma mensagem quando puder, só para saber se está tudo bem.",
    "Combinado então, te encontro lá na terça-feira à tarde.",
    "Vi que você não respondeu ainda, é para confirmar a reunião de amanhã?",
]

docs = spam + ham
labels = ["spam"] * len(spam) + ["ham"] * len(ham)

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(docs)
print(f"corpus: {len(docs)} mensagens ({len(spam)} spam, {len(ham)} ham), vocabulário = {len(vectorizer.vocabulary_)} termos")

clf = MultinomialNB()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pred = cross_val_predict(clf, X, labels, cv=cv)

print("\nrelatório do scikit-learn, validação cruzada em 5 dobras\n")
print(classification_report(labels, pred, digits=3))

print("matriz de confusão, linha é a classe real e coluna é a classe prevista")
print(f"{'':8s}{'ham':>8s}{'spam':>8s}")
for nome, linha in zip(["ham", "spam"], confusion_matrix(labels, pred, labels=["ham", "spam"])):
    print(f"{nome:8s}{linha[0]:8d}{linha[1]:8d}")

erros = [(docs[i], labels[i], pred[i]) for i in range(len(docs)) if labels[i] != pred[i]]
print(f"\nmensagens classificadas errado: {len(erros)}")
for texto, real, previsto in erros:
    print(f"  real={real:5s} previsto={previsto:5s}  {texto}")

**Análise:** O relatório do scikit-learn traz, para cada classe, a precisão, o recall e o f1-score, mais a acurácia geral. A precisão da classe spam é a fração das mensagens marcadas como spam que de fato eram spam. O recall da classe spam é a fração dos spams reais que o classificador marcou. A matriz de confusão mostra as duas de uma vez, e a lista abaixo dela traz a mensagem classificada errado, que mostra em que palavras a classificação se apoiou.

O classificador usa só a contagem de palavras, sem informação sobre o sentido delas.

---
## 2. TF-IDF, recuperação da passagem que responde à pergunta

Na contagem, a palavra que ocorre em quase todo documento recebe o maior valor e é a que menos separa um documento do outro. O TF-IDF pondera cada contagem pela raridade do termo no conjunto.

O cálculo desta seção é o do passo de recuperação de um RAG (retrieval-augmented generation, geração com contexto recuperado): dada uma pergunta, ordenar os trechos pelo cosseno e verificar em que posição fica o trecho correto.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Acervo de uma pagina interna sobre modelos de linguagem. O trecho 0 e o glossario,
# que repete os termos de todos os outros sem responder pergunta nenhuma.
trechos = [
    "Este glossário reúne os termos que aparecem no material sobre modelo de linguagem. O modelo recebe texto, o modelo divide o texto em token, o modelo representa cada palavra em um vetor, o modelo pondera as palavras da frase e o modelo escreve a resposta. Cada um desses passos do modelo tem uma página própria neste material, com a definição do termo, o exemplo de uso e o limite conhecido de cada técnica.",
    "O sistema RAG combina um passo de recuperação de documentos com um modelo generativo. A pergunta do usuário recupera os trechos mais próximos de um acervo, e esses trechos entram no prompt do modelo, que redige a resposta apoiada neles.",
    "O mecanismo de atenção permite que o modelo pondere a relevância de cada palavra de uma frase em relação às outras. Cada posição da frase recebe um peso, e é esse peso que decide quanto cada palavra contribui para a representação das demais.",
    "O treino de word embeddings aprende a representação de uma palavra a partir das palavras vizinhas. Palavras que aparecem em contextos parecidos terminam com vetores próximos, mesmo quando a grafia das duas não tem nada em comum.",
    "A tokenização quebra o texto em unidades menores que a palavra. O modelo não lê a frase como uma sequência de palavras, e sim como uma sequência de tokens, e palavra rara costuma ser dividida em vários deles.",
    "O ajuste fino reaproveita os pesos de um modelo já treinado e adapta o modelo a uma tarefa nova, com um conjunto de exemplos pequeno diante do usado no treino original.",
    "A quantização reduz a precisão numérica dos pesos do modelo. Com menos bits por peso, o modelo ocupa menos memória e responde mais rápido, ao custo de alguma perda de qualidade na resposta.",
    "A janela de contexto limita quantos tokens o modelo consegue considerar em uma única chamada. Texto que não cabe na janela precisa ser cortado ou resumido antes de entrar no prompt.",
    "Uma cesta de compras pode ser tratada como uma frase, e cada produto como uma palavra. Com isso o mesmo treino de embeddings aprende a representação de um produto a partir dos produtos comprados junto com ele.",
    "A alucinação acontece quando o modelo redige uma resposta fluente e sem apoio no texto que recebeu. O sintoma mais comum é a citação de um número ou de uma fonte que não existe no acervo consultado.",
]

perguntas = [
    ("O que o modelo faz no mecanismo de RAG?", 1),
    ("O que o modelo faz com as palavras da frase no mecanismo de atenção?", 2),
    ("Como o modelo aprende a representação de uma palavra?", 3),
    ("O que o modelo faz com o texto na tokenização?", 4),
    ("O que o ajuste fino faz com os pesos do modelo?", 5),
    ("Como o modelo passa a ocupar menos memória?", 6),
    ("O que limita quantos tokens o modelo considera em uma chamada?", 7),
    ("Como o modelo trata um produto como se fosse uma palavra?", 8),
    ("O que é uma alucinação do modelo?", 9),
]

# As duas vetorizacoes recebem os mesmos trechos e as mesmas perguntas.
# A unica diferenca entre elas e o peso que cada termo recebe.
vetorizadores = {"bag of words": CountVectorizer(), "TF-IDF": TfidfVectorizer()}

for nome, vetorizador in vetorizadores.items():
    X = vetorizador.fit_transform(trechos)
    acertos = 0
    print(f"=== {nome} ===")
    for pergunta, indice_correto in perguntas:
        q = vetorizador.transform([pergunta])
        sims = cosine_similarity(q, X)[0]
        topo = int(np.argsort(sims)[::-1][0])
        if topo == indice_correto:
            acertos += 1
            marca = "OK   "
        else:
            marca = "ERROU"
        print(f"  {marca} recuperou [{topo}] esperado [{indice_correto}] sim={sims[topo]:.3f}  {pergunta}")
    print(f"  acerto no topo 1: {acertos}/{len(perguntas)}\n")

**Análise:** a linha de acerto no topo 1 de cada vetorização, e qual trecho a contagem recupera nos erros dela.

O acervo é de um domínio só, e por isso quase todo trecho contém modelo, palavra, texto e frase. O trecho 0 é o glossário, que repete esses termos e não responde a nenhuma das perguntas. Na maior parte dos erros da contagem é ele que aparece no topo, porque soma muitas ocorrências dos termos que a pergunta também tem.

O TF-IDF divide o peso de cada termo pelo número de trechos em que ele aparece. Termo presente em quase todo trecho perde peso, a vantagem do glossário desaparece e os termos raros passam a decidir.

É este o cálculo que sustenta a busca de um sistema de RAG, antes de qualquer parte generativa entrar.

In [ ]:
# A pergunta que o TF-IDF erra, comparada com todos os trechos, mostra de onde
# vem a confusao da recuperacao.
PERGUNTA = "Como o modelo aprende a representação de uma palavra?"
ESPERADO = 3

vetorizador = TfidfVectorizer()
X = vetorizador.fit_transform(trechos)
q = vetorizador.transform([PERGUNTA])
sims = cosine_similarity(q, X)[0]
recuperado = int(np.argsort(sims)[::-1][0])

print(f"pergunta: {PERGUNTA}")
print(f"trecho que responde: [{ESPERADO}]\n")
print("similaridade de cada trecho com a pergunta, da maior para a menor")
for i in np.argsort(sims)[::-1]:
    if i == recuperado:
        marca = "  <- recuperado"
    elif i == ESPERADO:
        marca = "  <- responde a pergunta"
    else:
        marca = ""
    print(f"  [{i}] {sims[i]:.3f}  {trechos[i][:60]}...{marca}")

# Os termos da pergunta, o peso que o TF-IDF da a cada um e em qual dos dois
# trechos ele aparece.
analisador = vetorizador.build_analyzer()
vocabulario = vetorizador.vocabulary_
peso_recuperado = X[recuperado].toarray()[0]
peso_esperado = X[ESPERADO].toarray()[0]

print(f"\ntermos da pergunta e em qual dos dois trechos cada um aparece")
print(f"{'termo':16s}{'idf':>6s}{'[' + str(recuperado) + ']':>7s}{'[' + str(ESPERADO) + ']':>7s}")
ja_listados = []
for termo in analisador(PERGUNTA):
    if termo in ja_listados or termo not in vocabulario:
        continue
    ja_listados.append(termo)
    coluna = vocabulario[termo]
    if peso_recuperado[coluna] > 0:
        no_recuperado = "sim"
    else:
        no_recuperado = "-"
    if peso_esperado[coluna] > 0:
        no_esperado = "sim"
    else:
        no_esperado = "-"
    print(f"{termo:16s}{vetorizador.idf_[coluna]:6.2f}{no_recuperado:>7s}{no_esperado:>7s}")

**Análise:** a posição do trecho que responde a pergunta, e a tabela de termos.

Os dois trechos do topo contêm os mesmos termos raros da pergunta, que são aprende, representação e palavra. O trecho recuperado fala da representação de um produto tratado como palavra, e o trecho que responde fala da representação de uma palavra. A diferença entre os dois está no que as palavras querem dizer, e não em quais palavras aparecem, então nenhum peso calculado sobre contagem separa um do outro.

O `idf` da tabela é o peso que o TF-IDF dá a cada termo, e ele vem da contagem de trechos em que o termo aparece. É estatística do acervo, e não significado. É por isso que a próxima seção troca a contagem por uma representação aprendida.

---
## 3. Embeddings estáticos, proximidade entre palavras diferentes

O TF-IDF compara documentos pelas palavras em comum, e por isso enxerga a grafia e não o sentido. Um embedding estático atribui um vetor por palavra, aprendido a partir das palavras vizinhas no corpus de treino.

O modelo usado aqui é o word2vec CBOW de 50 dimensões do NILC, treinado em português, com 929.606 palavras no vocabulário. É a mesma arquitetura CBOW do slide da aula: as palavras vizinhas entram juntas e a rede é treinada para prever a palavra central.

O caso tem três frases. A frase A, a frase B do mesmo assunto de A e sem nenhuma palavra em comum com ela, e a frase C de assunto diferente que compartilha com A a palavra banco. A hipótese é que o TF-IDF coloque C mais perto de A do que B, porque compara pela palavra repetida, e que o embedding inverta essa ordem.

In [ ]:
from stop_words import get_stop_words
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file
import numpy as np
import re

# O NILC publica o mesmo corpus em 50, 100, 300, 600 e 1000 dimensoes, e tambem em
# GloVe, fastText e wang2vec. Trocar a linha abaixo troca o modelo.
MODELO_EMBEDDING = "nilc-nlp/word2vec-cbow-50d"

stopwords_pt = set(get_stop_words("portuguese"))

def tokenizar(frase):
    palavras = re.findall(r"[a-zà-ú]+", frase.lower())
    return [p for p in palavras if p not in stopwords_pt]

casos = [
    {
        "nome": "banco, instituição financeira contra assento de praça",
        "A": "O cliente foi ao banco tirar um empréstimo para reformar a casa.",
        "B": "O correntista solicitou financiamento para renovar o imóvel.",
        "C": "Ele sentou no banco da praça e observou as crianças brincando.",
    },
]

print("baixando o word2vec CBOW 50d do NILC, treinado em portugues (195 MB na primeira execucao)...")
emb_path = hf_hub_download(repo_id=MODELO_EMBEDDING, filename="embeddings.safetensors")
vocab_path = hf_hub_download(repo_id=MODELO_EMBEDDING, filename="vocab.txt")
data = load_file(emb_path)
vectors = data["embeddings"]
with open(vocab_path, encoding="utf-8") as f:
    vocab = [w.strip() for w in f]
indice = {palavra: i for i, palavra in enumerate(vocab)}

def vetor_frase(frase):
    tokens = [t for t in tokenizar(frase) if t in indice]
    if not tokens:
        return None
    return np.mean([vectors[indice[t]] for t in tokens], axis=0)

for caso in casos:
    print(f"\n=== {caso['nome']} ===")
    A, B, C = caso["A"], caso["B"], caso["C"]
    docs = [A, B, C]

    vec = TfidfVectorizer(tokenizer=tokenizar, lowercase=False, token_pattern=None)
    X = vec.fit_transform(docs)
    sim = cosine_similarity(X)
    sim_ab_tfidf, sim_ac_tfidf = sim[0, 1], sim[0, 2]

    vA, vB, vC = vetor_frase(A), vetor_frase(B), vetor_frase(C)
    sim_ab_emb = cosine_similarity([vA], [vB])[0, 0]
    sim_ac_emb = cosine_similarity([vA], [vC])[0, 0]

    print(f"TF-IDF     sim(A,B) = {sim_ab_tfidf:.3f}   sim(A,C) = {sim_ac_tfidf:.3f}   {'ERRA' if sim_ac_tfidf >= sim_ab_tfidf else 'acerta'}")
    print(f"embedding  sim(A,B) = {sim_ab_emb:.3f}   sim(A,C) = {sim_ac_emb:.3f}   {'acerta' if sim_ab_emb > sim_ac_emb else 'ERRA'}")

**Análise:** A similaridade por TF-IDF entre A e B, que não têm palavra em comum, e a similaridade entre A e C, que compartilham a palavra banco. Depois as mesmas duas similaridades pelo embedding.

O vetor de uma frase é a média dos vetores das palavras dela. O word2vec guarda um vetor por palavra inteira, e a palavra fora do vocabulário fica de fora da média.

A inversão depende de o corpus de treino trazer os dois sentidos da palavra em contextos distintos, e vale para o homônimo que satisfaz essa condição.

---
## 4. Self-attention, a palavra ganha contexto

Um embedding estático guarda uma linha por palavra, e por isso a mesma palavra recebe o mesmo vetor em qualquer frase. Na frase "Sentado no banco da praça vi um assalto a banco" as duas ocorrências de banco têm sentidos diferentes, e uma linha só atribui a mesma representação às duas.

Self-attention recalcula o vetor de cada palavra a partir das outras palavras da frase em que ela está. A saída de cada camada do modelo é um vetor por ocorrência, e não por palavra do vocabulário.

### 4.1 A mesma palavra, dois vetores

A célula abaixo passa por um BERT treinado em português duas frases em que banco aparece duas vezes. Na primeira as duas ocorrências têm sentidos diferentes, e na segunda têm o mesmo sentido. De cada frase saem dois vetores, um por ocorrência, e o número medido é o cosseno entre eles. Se o vetor dependesse só da grafia, os dois cossenos seriam 1.

In [ ]:
from transformers import AutoTokenizer, AutoModel
from transformers import logging as hf_logging
import numpy as np
import torch

hf_logging.set_verbosity_error()      # o relatorio de carga do modelo nao interessa em aula
hf_logging.disable_progress_bar()     # nem a barra de progresso da leitura dos pesos

modelo_nome = "neuralmind/bert-base-portuguese-cased"   # cerca de 440 MB na primeira execucao
tokenizer = AutoTokenizer.from_pretrained(modelo_nome)
model = AutoModel.from_pretrained(modelo_nome)
model.eval()


def vetor_da_palavra(frase, palavra):
    """Vetor que a ultima camada do BERT produz para uma palavra dentro de uma frase."""
    entrada = tokenizer(frase, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(entrada["input_ids"][0])
    posicoes = [i for i, t in enumerate(tokens) if t.lower() == palavra.lower()]
    with torch.no_grad():
        saida = model(**entrada).last_hidden_state[0]
    return saida[posicoes].mean(0).numpy()


def cosseno(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


# Duas ocorrencias da mesma grafia dentro de uma frase so. A funcao devolve um
# vetor por ocorrencia, em vez da media, para as duas poderem ser comparadas.
def vetores_de_cada_ocorrencia(frase, palavra):
    """Um vetor por ocorrência da palavra na frase, pela última camada do BERT."""
    entrada = tokenizer(frase, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(entrada["input_ids"][0])
    posicoes = []
    for indice in range(len(tokens)):
        if tokens[indice].lower() == palavra.lower():
            posicoes.append(indice)
    with torch.no_grad():
        saida = model(**entrada).last_hidden_state[0]
    vetores = []
    for indice in posicoes:
        vetores.append(saida[indice].numpy())
    return vetores


frase_dois_sentidos = "Sentei no banco da praça em frente ao banco onde tenho conta."
frase_um_sentido = "O banco aprovou o empréstimo e o banco cobrou a tarifa."

primeiro, segundo = vetores_de_cada_ocorrencia(frase_dois_sentidos, "banco")
print(f"frase com dois sentidos: {frase_dois_sentidos}")
print(f"  dimensão do vetor de cada ocorrência: {len(primeiro)}")
print(f"  cosseno entre a primeira e a segunda ocorrência: {cosseno(primeiro, segundo):.3f}\n")

primeiro, segundo = vetores_de_cada_ocorrencia(frase_um_sentido, "banco")
print(f"frase com um sentido só: {frase_um_sentido}")
print(f"  cosseno entre a primeira e a segunda ocorrência: {cosseno(primeiro, segundo):.3f}")

**Análise:** os dois cossenos, medidos entre ocorrências da mesma grafia dentro de uma frase só.

Quando as duas ocorrências de banco carregam sentidos diferentes, o cosseno entre elas cai. Quando as duas carregam o mesmo sentido, ele sobe. A grafia é idêntica nos dois casos, e o que muda é o resto da frase.

No word2vec da seção 3 os dois números seriam 1,000, porque a tabela tem uma linha só para banco e toda ocorrência lê a mesma linha.

### 4.2 Qual sentido a ocorrência tem

Duas frases servem de referência, uma com banco no sentido de assento e outra no sentido de instituição financeira. Cada frase de teste é comparada com as duas, pelo vetor da ocorrência de banco, e fica com a referência mais próxima.

A última coluna traz a mesma comparação pelo word2vec, para contraste.

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file

MODELO_EMBEDDING = "nilc-nlp/word2vec-cbow-50d"
tabela = load_file(hf_hub_download(MODELO_EMBEDDING, "embeddings.safetensors"))["embeddings"]
vocabulario = [w.strip() for w in open(hf_hub_download(MODELO_EMBEDDING, "vocab.txt"), encoding="utf-8")]
linha_banco = tabela[vocabulario.index("banco")]   # a unica linha que o word2vec tem para banco

referencias = {
    "assento":     "Sentei no banco da praça.",
    "instituição": "O banco aprovou o empréstimo.",
}
testes = [
    ("Esperei sentado no banco do parque.", "assento"),
    ("Abri uma conta no banco ontem.", "instituição"),
    ("O banco de madeira estava molhado de chuva.", "assento"),
    ("A fila do banco estava cheia de correntistas.", "instituição"),
]

vetor_ref = {sentido: vetor_da_palavra(frase, "banco") for sentido, frase in referencias.items()}

print(f"{'BERT':>46s}  {'word2vec':>17s}")
print(f"{'frase de teste':46s}  {'assento':>8s}{'instit.':>9s}  {'assento':>8s}{'instit.':>9s}   escolha")
acertos = 0
for frase, sentido_certo in testes:
    v = vetor_da_palavra(frase, "banco")
    b_a, b_i = cosseno(v, vetor_ref["assento"]), cosseno(v, vetor_ref["instituição"])
    w_a = w_i = cosseno(linha_banco, linha_banco)   # a mesma linha nos dois lados
    escolha = "assento" if b_a > b_i else "instituição"
    acertos += escolha == sentido_certo
    print(f"{frase[:46]:46s}  {b_a:8.3f}{b_i:9.3f}  {w_a:8.3f}{w_i:9.3f}   {escolha}")

print(f"\nBERT escolheu o sentido correto em {acertos} das {len(testes)} frases.")
print("Pelo word2vec as duas colunas valem 1,000 em toda frase, e não há escolha a fazer.")


**Análise:** As duas colunas do BERT, que separam os sentidos, e as duas do word2vec, que empatam em 1,000 em toda linha.

O empate acontece porque o word2vec devolve a mesma linha para as seis ocorrências de banco. A representação estática não expressa a diferença que a pergunta pede, e por isso o método não erra a resposta, ele não consegue produzir resposta nenhuma.

### 4.3 De onde vem o contexto, os pesos de self-attention

As duas células anteriores mostram que o vetor muda com a frase. Esta mostra o mecanismo que produz a mudança, que é o peso que cada palavra dá às outras.

O material de Transformers da disciplina de NLP afirma que self-attention resolve "ele" para "panetone" e não para "forno", na frase "O panetone saiu do forno e ele estava muito gostoso", sem mostrar o número. A célula extrai os pesos de self-attention de um BERT treinado em português e mede a diferença.

In [ ]:
from transformers import AutoTokenizer, AutoModel
from transformers import logging as hf_logging
import torch

hf_logging.set_verbosity_error()      # o relatorio de carga do modelo nao interessa em aula
hf_logging.disable_progress_bar()     # nem a barra de progresso da leitura dos pesos

modelo_nome = "neuralmind/bert-base-portuguese-cased"   # cerca de 440 MB na primeira execucao
tokenizer = AutoTokenizer.from_pretrained(modelo_nome)
model = AutoModel.from_pretrained(modelo_nome, output_attentions=True)
model.eval()

frase = "O panetone saiu do forno e ele estava muito gostoso"
entrada = tokenizer(frase, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(entrada["input_ids"][0])
print("tokens:", tokens)

with torch.no_grad():
    saida = model(**entrada)
atencoes = saida.attentions
n_camadas, n_heads = len(atencoes), atencoes[0].shape[1]
print(f"{n_camadas} camadas x {n_heads} heads = {n_camadas*n_heads} combinações de self-attention")

idx_ele = tokens.index("ele")
idx_panetone = [2, 3, 4]   # 'pan', '##eto', '##ne'
idx_forno = [7, 8]         # 'forn', '##o'

melhores = []
for camada in range(n_camadas):
    att_camada = atencoes[camada][0]
    for head in range(att_camada.shape[0]):
        pp = att_camada[head, idx_ele, idx_panetone].mean().item()
        pf = att_camada[head, idx_ele, idx_forno].mean().item()
        melhores.append((pp - pf, camada, head, pp, pf))

melhores.sort(reverse=True)
n_favor = sum(1 for gap, *_ in melhores if gap > 0)
print(f"\nde {len(melhores)} combinações camada x head, {n_favor} favorecem panetone sobre forno ({n_favor/len(melhores):.1%})")

print("\ntop 3 heads que mais separam panetone de forno:")
for gap, camada, head, pp, pf in melhores[:3]:
    print(f"  camada {camada:2d} head {head:2d}: ele->panetone={pp:.3f}  ele->forno={pf:.3f}  diferença={gap:.3f}")

**Análise:** Em quantas das 144 combinações de camada e head o peso de "ele" sobre "panetone" supera o peso sobre "forno", e qual é o head em que a diferença é maior.

A média sobre as 144 combinações fica perto da metade. Boa parte da atenção, em muitos heads, vai para os tokens especiais [CLS] e [SEP], comportamento documentado por Clark et al. (2019). Existem heads específicos em que a separação é grande, e a medição é o que os identifica. É esta a correção do que o material da outra disciplina afirma sem medir.

---
## 5. Modelo pré-treinado, os mesmos dois problemas por instrução

As seções 1 e 2 pediram um vetorizador e um classificador treinado. Esta seção refaz as duas tarefas com um modelo generativo consumido por API, a partir de uma instrução escrita em português e sem nenhum treino.

A célula procura a chave em três lugares, nesta ordem: a variável de ambiente `ANTHROPIC_API_KEY`, o painel de segredos do Colab com esse mesmo nome, e por fim um campo digitado na hora, que não mostra o que você escreve.

In [ ]:
import os
from getpass import getpass

MODELO = "claude-sonnet-5"   # troque aqui para usar outro modelo

chave = os.environ.get("ANTHROPIC_API_KEY")
try:
    from google.colab import userdata          # no Colab, painel de segredos
    chave = chave or userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass
if not chave:
    # getpass nao ecoa o que e digitado, entao a chave nao fica na saida da celula
    chave = getpass("Cole a chave da API e tecle enter: ").strip()

mensagem_teste = "Você é um dos sorteados! Ligue agora para resgatar seu prêmio de R$5.000."
pergunta_teste = "O que é o mecanismo de RAG?"

prompt_classificacao = (
    'Classifique a mensagem abaixo como "spam" ou "normal". '
    "Responda com uma única palavra, sem pontuação e sem explicação.\n\n"
    f"Mensagem: {mensagem_teste}"
)
prompt_resposta = (
    "Responda à pergunta usando só os trechos abaixo. "
    "Cite entre colchetes o número do trecho que usou.\n\nTrechos:\n"
    + "\n".join(f"[{i}] {t}" for i, t in enumerate(trechos))
    + f"\n\nPergunta: {pergunta_teste}"
)

def responder(prompt, max_tokens=300):
    import anthropic
    cliente = anthropic.Anthropic(api_key=chave)
    r = cliente.messages.create(
        model=MODELO, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return next(b.text for b in r.content if b.type == "text").strip()

if not chave:
    print("Sem chave, nenhuma chamada é feita. Os dois prompts abaixo seriam enviados ao modelo.")
    print("\n=== prompt de classificação ===\n" + prompt_classificacao)
    print("\n=== prompt de recuperação e resposta ===\n" + prompt_resposta)
else:
    print("=== 1. o filtro de spam da seção 1, refeito por instrução ===")
    print("mensagem:", mensagem_teste)
    print("resposta do modelo:", responder(prompt_classificacao, 20))

    print("\n=== 2. a recuperação da seção 2, refeita por instrução ===")
    print("pergunta:", pergunta_teste)
    print("resposta do modelo:", responder(prompt_resposta, 300))


**Análise:** O que desapareceu do código em relação às seções 1 e 2, e o que passou a ser necessário no lugar. A comparação é de esforço e de dependências, e não de acurácia.

O trabalho sai da construção do modelo e vai para a descrição da tarefa, a escolha do modelo e a avaliação do resultado.

---
## 6. Consumo do modelo por API

As seções anteriores carregaram modelos na própria máquina. Um modelo pré-treinado consumido por API fica no servidor de quem o treinou, e o que trafega é a mensagem enviada e o texto devolvido.

Esta seção percorre quatro pontos: o que a chamada leva e o que ela traz, o reenvio da conversa, os dois parâmetros que controlam a variação da saída, e o contrato de formato que permite a um programa consumir a resposta.

### 6.1 Anatomia da chamada

A chamada leva o nome do modelo, a instrução de sistema, a lista de mensagens e os parâmetros. A resposta traz o texto, o motivo pelo qual a geração parou e a contagem de tokens de entrada e de saída, que é a unidade em que o uso é cobrado.

In [ ]:
import os
from getpass import getpass
import anthropic

MODELO = "claude-sonnet-5"   # troque aqui para usar outro modelo

chave = os.environ.get("ANTHROPIC_API_KEY")
try:
    from google.colab import userdata          # no Colab, painel de segredos
    chave = chave or userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass
if not chave:
    # getpass nao ecoa o que e digitado, entao a chave nao fica na saida da celula
    chave = getpass("Cole a chave da API e tecle enter: ").strip()

cliente = anthropic.Anthropic(api_key=chave)


def texto_da(resposta):
    """Primeiro bloco de texto da resposta. Alguns modelos devolvem outros blocos antes dele."""
    blocos = [b.text for b in resposta.content if b.type == "text"]
    return blocos[0] if blocos else "(a resposta terminou antes do bloco de texto)"

SISTEMA = "Você responde em português, de forma direta."
PERGUNTA = "Em duas frases, o que é um banco de dados vetorial?"

resposta = cliente.messages.create(
    model=MODELO,
    max_tokens=300,          # teto da saida: ao chegar nele o texto e cortado, e nao resumido
    system=SISTEMA,
    messages=[{"role": "user", "content": PERGUNTA}],
)

print("texto devolvido:")
print(texto_da(resposta))
print(f"\nmotivo da parada: {resposta.stop_reason}")
print(f"tokens de entrada: {resposta.usage.input_tokens}")
print(f"tokens de saída:   {resposta.usage.output_tokens}")

# A mesma chamada com um teto baixo, para ver o que acontece no limite.
curta = cliente.messages.create(
    model=MODELO,
    max_tokens=20,
    system=SISTEMA,
    messages=[{"role": "user", "content": PERGUNTA}],
)
print(f"\ncom max_tokens=20 o texto devolvido é: {texto_da(curta)!r}")
print(f"motivo da parada: {curta.stop_reason}")


**Análise:** As duas contagens de token da primeira chamada, que é o que a fatura mede, e o motivo da parada nas duas chamadas.

Na primeira a geração para porque o modelo terminou a resposta. Na segunda ela para porque bateu no teto, e o texto chega cortado no meio. O `max_tokens` limita a saída, e não pede ao modelo que caiba nele.

### 6.2 Reenvio da conversa a cada chamada

Cada chamada é independente. O servidor não guarda o que foi dito antes, e a continuidade de uma conversa existe porque o programa reenvia o histórico a cada nova mensagem.

In [ ]:
PRIMEIRA = "Meu nome é Thiago e dou aula de modelos de linguagem."
SEGUIMENTO = "Qual é o meu nome?"

# 1. o seguimento sozinho, sem nada antes dele
sozinho = cliente.messages.create(
    model=MODELO, max_tokens=400,
    messages=[{"role": "user", "content": SEGUIMENTO}],
)

# 2. o mesmo seguimento, com as duas mensagens anteriores reenviadas junto
primeira = cliente.messages.create(
    model=MODELO, max_tokens=400,
    messages=[{"role": "user", "content": PRIMEIRA}],
)
com_historico = cliente.messages.create(
    model=MODELO, max_tokens=400,
    messages=[
        {"role": "user", "content": PRIMEIRA},
        {"role": "assistant", "content": texto_da(primeira)},
        {"role": "user", "content": SEGUIMENTO},
    ],
)

print("sem histórico")
print(f"  resposta: {texto_da(sozinho).strip()[:160]}")
print(f"  tokens de entrada: {sozinho.usage.input_tokens}")
print("\ncom histórico reenviado")
print(f"  resposta: {texto_da(com_historico).strip()[:160]}")
print(f"  tokens de entrada: {com_historico.usage.input_tokens}")


**Análise:** As duas respostas e as duas contagens de tokens de entrada.

Sem histórico o modelo não tem como saber o nome, porque a informação não foi enviada. Com o histórico reenviado ele responde, e os tokens de entrada sobem, porque a conversa inteira viaja de novo em cada chamada. Numa conversa longa esse custo cresce a cada turno.

### 6.3 Temperatura e top_p

Os dois parâmetros controlam quanta variação existe entre execuções da mesma chamada. A temperatura achata ou acentua as diferenças entre as probabilidades dos tokens candidatos. O top_p corta a cauda e deixa disponíveis para sorteio apenas os tokens mais prováveis, até somar a fração indicada.

A geração mais recente dos modelos retirou os dois. Em Claude Sonnet 5 e Opus 5, enviar `temperature`, `top_p` ou `top_k` com valor diferente do padrão devolve erro 400, e a documentação orienta a controlar o comportamento pelo prompt de sistema. A célula abaixo roda a demonstração num modelo da geração anterior, que ainda aceita os parâmetros, e no fim mede a recusa do modelo atual.

A pergunta pede um nome curto, porque um espaço de resposta pequeno torna contável quantas respostas diferentes apareceram. Com um texto longo, duas execuções sempre diferem e a contagem não diria nada.

In [ ]:
import anthropic

# O SDK 1.8 tirou temperature e top_p da assinatura de messages.create,
# entao os dois viajam em extra_body. E a geracao 5 dos modelos nao os aceita
# mais, por isso a demonstracao roda num modelo da geracao anterior.
MODELO_AMOSTRAGEM = "claude-sonnet-4-5-20250929"

SISTEMA_NOME = "Você nomeia produtos financeiros. Responda só o nome, em até 3 palavras."
PEDIDO = "Dê um nome comercial para uma conta digital para jovens."
N = 5   # repeticoes por configuracao

def nomes(parametros):
    saidas = []
    for _ in range(N):
        r = cliente.messages.create(
            model=MODELO_AMOSTRAGEM, max_tokens=20, system=SISTEMA_NOME,
            messages=[{"role": "user", "content": PEDIDO}], extra_body=parametros)
        saidas.append(texto_da(r).strip().lower().strip("*").rstrip("."))
    return saidas

# A API recusa temperature e top_p na mesma chamada, entao cada um roda sozinho.
configuracoes = [
    ("temperature=0.0", {"temperature": 0.0}),
    ("temperature=1.0", {"temperature": 1.0}),
    ("top_p=0.1", {"top_p": 0.1}),
    ("top_p=1.0", {"top_p": 1.0}),
]

print(f"modelo usado nesta célula: {MODELO_AMOSTRAGEM}\n")
for rotulo, parametros in configuracoes:
    saidas = nomes(parametros)
    print(f"{rotulo}  ->  {len(set(saidas))} resposta(s) diferente(s) em {N} execuções")
    for s in saidas:
        print(f"     {s}")
    print()

# A mesma chamada contra o modelo atual, para ver a recusa.
print(f"a mesma chamada com temperature contra {MODELO}:")
try:
    cliente.messages.create(
        model=MODELO, max_tokens=20, system=SISTEMA_NOME,
        messages=[{"role": "user", "content": PEDIDO}], extra_body={"temperature": 0.0})
    print("   aceita")
except anthropic.BadRequestError as erro:
    print(f"   recusada, {erro.status_code}: {erro.body['error']['message']}")


**Análise:** A contagem de respostas diferentes em cada configuração, e a recusa impressa na última linha.

Valor baixo de temperatura ou de top_p concentra o sorteio nos tokens mais prováveis, e a saída se repete. Valor alto abre o leque, e a saída varia. Um sistema que precisa de resposta estável trabalha com valor baixo, e um que procura alternativas trabalha com valor alto. Temperatura zero reduz a variação, e não a elimina por completo.

**O estado do parâmetro nos três maiores provedores**, conferido em 24/09/2026. Os três caminham na mesma direção, à medida que os modelos passam a raciocinar antes de responder, e cada um trata o parâmetro de um jeito.

| provedor | situação |
|---|---|
| Anthropic, Claude Sonnet 5 e Opus 5 | recusam. Valor diferente do padrão devolve erro 400, e a orientação é usar o prompt de sistema |
| Google, Gemini 3 | aceita, e a documentação recomenda manter a temperatura no padrão 1,0, porque valor menor pode causar repetição em cadeia e perda de desempenho em tarefa de raciocínio |
| OpenAI, modelos de raciocínio | a documentação orienta remover `temperature` e `top_p` quando o esforço de raciocínio é diferente de `none` |

Quem mantém código que envia esses parâmetros precisa conferir o comportamento a cada troca de modelo.

Fontes: documentação do Claude Sonnet 5 e do Gemini 3, e o guia do modelo mais recente da OpenAI, todas consultadas em 24/09/2026.

### 6.4 Contrato de saída

Um programa consome a resposta quando ela chega num formato previsto. A instrução abaixo pede JSON com campos nomeados, e a medida é quantas das respostas a função `json.loads` lê sem tratamento.

In [ ]:
import json
import re

MENSAGENS = [
    "Clique no link e ganhe o prêmio.",
    "Confirmo a reunião de amanhã e levo o relatório.",
    "PARABÉNS! Você foi sorteado, ligue agora para resgatar.",
    "Bom dia, o boleto de setembro já está disponível.",
    "Crédito liberado sem consulta, responda SIM.",
    "Te encontro na recepção às nove horas.",
    "Última chance de resgatar seus pontos, informe o código.",
    "Segue em anexo a planilha que você pediu.",
]

SISTEMA_JSON = (
    "Você classifica mensagens recebidas por um provedor de e-mail.\n"
    "Responda em JSON com os campos classe (spam ou normal), "
    "confianca (número entre 0 e 1) e motivo (até 60 caracteres)."
)

brutos = []
for m in MENSAGENS:
    r = cliente.messages.create(
        model=MODELO, max_tokens=200,
        system=SISTEMA_JSON, messages=[{"role": "user", "content": f"Mensagem: {m}"}])
    brutos.append(texto_da(r).strip())

def tenta_ler(texto):
    try:
        return json.loads(texto)
    except Exception:
        return None

# A cerca de codigo e o envolucro mais comum que impede a leitura direta.
sem_cerca = [re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S) for t in brutos]

leu_direto = sum(tenta_ler(t) is not None for t in brutos)
leu_sem_cerca = sum(tenta_ler(t) is not None for t in sem_cerca)

for m, t in zip(MENSAGENS, brutos):
    obj = tenta_ler(t) or tenta_ler(re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S))
    if obj:
        print(f"  {obj.get('classe','?'):7s} {obj.get('confianca','?')!s:5s}  {m[:48]}")
    else:
        print(f"  {'ilegível':7s} {'':5s}  {m[:48]}")

print(f"\nrespostas lidas por json.loads sem tratamento: {leu_direto} de {len(MENSAGENS)}")
print(f"respostas lidas depois de remover a cerca de código: {leu_sem_cerca} de {len(MENSAGENS)}")


**Análise:** As duas contagens do fim. A primeira diz quantas respostas um programa consumiria como chegaram, e a segunda quantas passam depois de remover a cerca de código que o modelo costuma colocar em volta do JSON.

A diferença entre as duas é o tratamento que todo sistema em produção acaba escrevendo. Pedir o formato na instrução resolve a maior parte, e o restante é código de limpeza.

O acesso por API dá ao programa a capacidade do modelo, e o contrato de saída é o que torna a resposta consumível. Os dois juntos são o que permite construir um sistema sobre um modelo que você não treinou.